# Assignment 07: IMDB Text Classification in Multiple Ways

**Student:** Amankeldy Abylaj  
**Task:** Classify IMDB reviews as positive or negative using several text representations.

This notebook compares:

1. Word-level integer encoding + learned embedding + GRU  
2. Gensim Word2Vec embeddings + classical classifier  
3. Character-level neural model  
4. Different preprocessing and OOV strategies


In [ ]:
# ============================================================
# Setup cell
# Run this cell first in Google Colab / Jupyter.
# If gensim gives a NumPy/SciPy compatibility error, run this cell,
# restart the runtime, and then run all cells again.
# ============================================================

import sys
import subprocess
import importlib.util

def install_if_missing(package, pip_name=None):
    if pip_name is None:
        pip_name = package
    if importlib.util.find_spec(package) is None:
        print(f"Installing {pip_name} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])

install_if_missing("datasets")
install_if_missing("gensim")
install_if_missing("spacy")
install_if_missing("sklearn", "scikit-learn")

print("Setup check finished.")

In [ ]:
# Core imports

import os
import re
import math
import random
import string
from collections import Counter

import numpy as np
import pandas as pd

from datasets import load_dataset
from gensim.models import Word2Vec

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import spacy
from spacy.lang.en.stop_words import STOP_WORDS

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

## Part A. Load IMDB and Compare Preprocessing

In this part, I load the real IMDB dataset, create train/validation/test splits, inspect the dataset, and compare two preprocessing variants.

The goal of Part A is not training yet.  
The goal is to understand and prepare the text before it is passed into models.


In [ ]:
# Load real IMDB dataset from Hugging Face
# This is the real dataset, not a handmade fallback.

imdb = load_dataset("imdb")

train_full_texts = list(imdb["train"]["text"])
train_full_labels = list(imdb["train"]["label"])

test_full_texts = list(imdb["test"]["text"])
test_full_labels = list(imdb["test"]["label"])

# Create validation split from the original train set
idx = np.arange(len(train_full_texts))
np.random.shuffle(idx)

val_ratio = 0.2
val_size = int(len(idx) * val_ratio)

val_idx = idx[:val_size]
train_idx = idx[val_size:]

train_texts = [train_full_texts[i] for i in train_idx]
train_labels = [train_full_labels[i] for i in train_idx]

val_texts = [train_full_texts[i] for i in val_idx]
val_labels = [train_full_labels[i] for i in val_idx]

# To make the notebook run faster, I use a test subset.
# The split is still taken from the real IMDB test set.
TEST_LIMIT = 10000
test_texts = test_full_texts[:TEST_LIMIT]
test_labels = test_full_labels[:TEST_LIMIT]

print("Split sizes:")
print("Train:", len(train_texts))
print("Validation:", len(val_texts))
print("Test:", len(test_texts))

In [ ]:
def average_length(texts):
    return np.mean([len(t.split()) for t in texts])

def class_balance(labels):
    c = Counter(labels)
    total = len(labels)
    return {label: (count, round(count / total, 4)) for label, count in c.items()}

print("Average review length:")
print("Train:", round(average_length(train_texts), 2))
print("Validation:", round(average_length(val_texts), 2))
print("Test:", round(average_length(test_texts), 2))

print("\nClass balance:")
print("Train:", class_balance(train_labels))
print("Validation:", class_balance(val_labels))
print("Test:", class_balance(test_labels))

print("\n2 positive examples:")
pos_printed = 0
for text, label in zip(train_texts, train_labels):
    if label == 1 and pos_printed < 2:
        print("POS:", text[:350].replace("\n", " "), "...")
        pos_printed += 1

print("\n2 negative examples:")
neg_printed = 0
for text, label in zip(train_texts, train_labels):
    if label == 0 and neg_printed < 2:
        print("NEG:", text[:350].replace("\n", " "), "...")
        neg_printed += 1

In [ ]:
# Preprocessing variant 1:
# lowercase + punctuation removal + whitespace split

punct_table = str.maketrans("", "", string.punctuation)

def preprocess_simple(text):
    text = text.lower()
    text = text.translate(punct_table)
    text = re.sub(r"\s+", " ", text).strip()
    return text.split()


# Preprocessing variant 2:
# spaCy tokenizer + lowercase + only alphabetic tokens + stopword removal
# Important: keep negation words because they matter for sentiment.

nlp = spacy.blank("en")
NEGATION_WORDS = {"no", "not", "never", "nor", "n't"}

def preprocess_advanced(text):
    doc = nlp(text.lower())
    tokens = []
    for tok in doc:
        t = tok.text.strip()
        if not t.isalpha():
            continue
        if t in STOP_WORDS and t not in NEGATION_WORDS:
            continue
        tokens.append(t)
    return tokens


# Tokenize all splits once
train_tokens_simple = [preprocess_simple(t) for t in train_texts]
val_tokens_simple = [preprocess_simple(t) for t in val_texts]
test_tokens_simple = [preprocess_simple(t) for t in test_texts]

train_tokens_advanced = [preprocess_advanced(t) for t in train_texts]
val_tokens_advanced = [preprocess_advanced(t) for t in val_texts]
test_tokens_advanced = [preprocess_advanced(t) for t in test_texts]

print("Tokenization finished.")

In [ ]:
def preprocessing_report(name, tokenized_texts):
    all_tokens = [tok for review in tokenized_texts for tok in review]
    counter = Counter(all_tokens)
    vocab_size = len(counter)
    avg_tokens = np.mean([len(x) for x in tokenized_texts])

    print(f"\n{name}")
    print("Average tokens per review:", round(avg_tokens, 2))
    print("Vocabulary size before cutoff:", vocab_size)
    print("10 most frequent tokens:", counter.most_common(10))

    rare = sorted([w for w, c in counter.items() if c == 1])[:10]
    print("10 rare tokens:", rare)


preprocessing_report("Variant 1: simple lowercase + punctuation removal", train_tokens_simple)
preprocessing_report("Variant 2: tokenizer + stopword removal + keep negation", train_tokens_advanced)

print("\n5 tokenized examples:")
for i in range(5):
    print(f"Example {i+1}")
    print("Simple:", train_tokens_simple[i][:25])
    print("Advanced:", train_tokens_advanced[i][:25])
    print("-" * 80)

### Part A explanation

Simple preprocessing keeps more information, including stopwords and negation words. This can help sentiment classification because words like **not**, **no**, and **never** can completely change the meaning.

Advanced preprocessing removes many common words, so the vocabulary becomes smaller and cleaner. However, if preprocessing is too aggressive, the model may lose important sentiment cues. Therefore, preprocessing is not always automatically better when it removes more words.


## Part B. Word-Level Integer Encoding Model

In this part, I build the first neural model.

Pipeline:

**text → tokens → word ids → embedding → GRU → classifier**

The vocabulary is built only from the training split to avoid data leakage.


In [ ]:
# Build vocabulary from TRAIN ONLY

PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"
PAD_IDX = 0
UNK_IDX = 1

min_freq = 3
max_len_word = 220

word_counter = Counter(tok for review in train_tokens_simple for tok in review)

stoi = {
    PAD_TOKEN: PAD_IDX,
    UNK_TOKEN: UNK_IDX
}

# Sort for reproducibility
for word, count in sorted(word_counter.items()):
    if count >= min_freq:
        stoi[word] = len(stoi)

itos = {i: w for w, i in stoi.items()}

print("Word model setup:")
print("min_freq:", min_freq)
print("maximum sequence length:", max_len_word)
print("vocabulary size:", len(stoi))
print("padding rule: post-padding with <PAD>")
print("truncation rule: cut tokens after max_len")

In [ ]:
def encode_word_tokens(tokens, stoi, max_len):
    ids = [stoi.get(tok, UNK_IDX) for tok in tokens]

    # Real length is needed for pack_padded_sequence.
    length = min(len(ids), max_len)
    if length == 0:
        length = 1

    if len(ids) < max_len:
        ids = ids + [PAD_IDX] * (max_len - len(ids))
    else:
        ids = ids[:max_len]

    return ids, length


X_train_word, len_train_word = zip(*[encode_word_tokens(x, stoi, max_len_word) for x in train_tokens_simple])
X_val_word, len_val_word = zip(*[encode_word_tokens(x, stoi, max_len_word) for x in val_tokens_simple])
X_test_word, len_test_word = zip(*[encode_word_tokens(x, stoi, max_len_word) for x in test_tokens_simple])

y_train = np.array(train_labels, dtype=np.int64)
y_val = np.array(val_labels, dtype=np.int64)
y_test = np.array(test_labels, dtype=np.int64)

print("Encoded train shape:", np.array(X_train_word).shape)
print("Encoded validation shape:", np.array(X_val_word).shape)
print("Encoded test shape:", np.array(X_test_word).shape)

In [ ]:
class WordDataset(Dataset):
    def __init__(self, X, lengths, y):
        self.X = torch.tensor(X, dtype=torch.long)
        self.lengths = torch.tensor(lengths, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.lengths[idx], self.y[idx]


class WordClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_size, num_classes, pad_idx):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.rnn = nn.GRU(embed_dim, hidden_size, batch_first=True)
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x, lengths):
        embedded = self.embedding(x)

        packed = nn.utils.rnn.pack_padded_sequence(
            embedded,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        _, h_n = self.rnn(packed)
        last_hidden = h_n[-1]
        last_hidden = self.dropout(last_hidden)

        return self.fc(last_hidden)


batch_size = 64

train_loader_word = DataLoader(
    WordDataset(X_train_word, len_train_word, y_train),
    batch_size=batch_size,
    shuffle=True
)

val_loader_word = DataLoader(
    WordDataset(X_val_word, len_val_word, y_val),
    batch_size=batch_size,
    shuffle=False
)

test_loader_word = DataLoader(
    WordDataset(X_test_word, len_test_word, y_test),
    batch_size=batch_size,
    shuffle=False
)

word_model = WordClassifier(
    vocab_size=len(stoi),
    embed_dim=100,
    hidden_size=128,
    num_classes=2,
    pad_idx=PAD_IDX
).to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(word_model.parameters(), lr=0.001)

print(word_model)

In [ ]:
def train_one_epoch_word(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for x, lengths, y in loader:
        x = x.to(DEVICE)
        lengths = lengths.to(DEVICE)
        y = y.to(DEVICE)

        optimizer.zero_grad()
        logits = model(x, lengths)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * y.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == y).sum().item()
        total += y.size(0)

    return total_loss / total, correct / total


@torch.no_grad()
def evaluate_word(model, loader, criterion=None):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    all_preds = []
    all_targets = []

    for x, lengths, y in loader:
        x = x.to(DEVICE)
        lengths = lengths.to(DEVICE)
        y = y.to(DEVICE)

        logits = model(x, lengths)

        if criterion is not None:
            loss = criterion(logits, y)
            total_loss += loss.item() * y.size(0)

        preds = logits.argmax(dim=1)

        correct += (preds == y).sum().item()
        total += y.size(0)

        all_preds.extend(preds.cpu().numpy().tolist())
        all_targets.extend(y.cpu().numpy().tolist())

    avg_loss = total_loss / total if criterion is not None else None
    acc = correct / total

    return avg_loss, acc, np.array(all_preds), np.array(all_targets)


# Train the GRU model
epochs = 4

history_word = []

for epoch in range(1, epochs + 1):
    train_loss, train_acc = train_one_epoch_word(word_model, train_loader_word, optimizer, criterion)
    val_loss, val_acc, _, _ = evaluate_word(word_model, val_loader_word, criterion)

    history_word.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "val_loss": val_loss,
        "val_acc": val_acc
    })

    print(
        f"Epoch {epoch}/{epochs} | "
        f"train_loss={train_loss:.4f}, train_acc={train_acc:.4f}, "
        f"val_loss={val_loss:.4f}, val_acc={val_acc:.4f}"
    )

train_loss_word, train_acc_word, train_preds_word, train_targets_word = evaluate_word(word_model, train_loader_word, criterion)
val_loss_word, val_acc_word, val_preds_word, val_targets_word = evaluate_word(word_model, val_loader_word, criterion)
test_loss_word, test_acc_word, test_preds_word, test_targets_word = evaluate_word(word_model, test_loader_word, criterion)

print("\nWord-level model accuracy:")
print("Train accuracy:", round(train_acc_word, 4))
print("Validation accuracy:", round(val_acc_word, 4))
print("Test accuracy:", round(test_acc_word, 4))

In [ ]:
# Plot training curves for Part B

import matplotlib.pyplot as plt

history_df_word = pd.DataFrame(history_word)

plt.figure(figsize=(7, 4))
plt.plot(history_df_word["epoch"], history_df_word["train_acc"], marker="o", label="train_acc")
plt.plot(history_df_word["epoch"], history_df_word["val_acc"], marker="o", label="val_acc")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Part B: Word GRU Accuracy")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Show 3 correct and 3 incorrect predictions from the test set

correct_idx = [i for i, (p, y) in enumerate(zip(test_preds_word, test_targets_word)) if p == y][:3]
wrong_idx = [i for i, (p, y) in enumerate(zip(test_preds_word, test_targets_word)) if p != y][:3]

print("3 correct predictions:")
for i in correct_idx:
    print(f"Index {i} | pred={test_preds_word[i]} | true={test_targets_word[i]}")
    print(test_texts[i][:320].replace("\n", " "), "...")
    print("-" * 80)

print("\n3 incorrect predictions:")
for i in wrong_idx:
    print(f"Index {i} | pred={test_preds_word[i]} | true={test_targets_word[i]}")
    print(test_texts[i][:320].replace("\n", " "), "...")
    print("-" * 80)

### Part B interpretation

The word-level GRU model uses a learned embedding layer, so word vectors are optimized specifically for sentiment classification. The GRU also reads tokens as a sequence, which means it can use word order better than a simple bag-of-words model.

This model is expected to produce realistic accuracy, not 1.0. On real IMDB data, a small GRU usually reaches around 0.80–0.85 test accuracy depending on training settings.


## Part C. Gensim Word Embedding Model

In this part, I train Word2Vec on training reviews only. Then each review is represented by one vector using mean pooling.

I compare two OOV strategies:

1. **skip**: ignore words missing from Word2Vec vocabulary  
2. **avg**: replace missing words with the global average vector


In [ ]:
# Train Word2Vec on training tokens only

w2v_dim = 100

w2v = Word2Vec(
    sentences=train_tokens_simple,
    vector_size=w2v_dim,
    window=5,
    min_count=2,
    workers=4,
    sg=1,
    epochs=10,
    seed=SEED
)

global_avg_vec = np.mean(w2v.wv.vectors, axis=0)

def review_to_vec(tokens, strategy="skip"):
    vectors = []
    found = 0
    missing = 0

    for tok in tokens:
        if tok in w2v.wv:
            vectors.append(w2v.wv[tok])
            found += 1
        else:
            missing += 1
            if strategy == "avg":
                vectors.append(global_avg_vec)

    if len(vectors) == 0:
        return np.zeros(w2v_dim), found, missing

    return np.mean(vectors, axis=0), found, missing


def build_review_matrix(tokenized_texts, strategy):
    X = []
    found_total = 0
    missing_total = 0

    for tokens in tokenized_texts:
        vec, found, missing = review_to_vec(tokens, strategy=strategy)
        X.append(vec)
        found_total += found
        missing_total += missing

    return np.vstack(X), found_total, missing_total


gensim_results = []

for strategy in ["skip", "avg"]:
    X_train_g, found_train, missing_train = build_review_matrix(train_tokens_simple, strategy)
    X_val_g, found_val, missing_val = build_review_matrix(val_tokens_simple, strategy)
    X_test_g, found_test, missing_test = build_review_matrix(test_tokens_simple, strategy)

    clf = LogisticRegression(max_iter=1000, random_state=SEED)
    clf.fit(X_train_g, y_train)

    train_pred = clf.predict(X_train_g)
    val_pred = clf.predict(X_val_g)
    test_pred = clf.predict(X_test_g)

    gensim_results.append({
        "strategy": strategy,
        "train_acc": accuracy_score(y_train, train_pred),
        "val_acc": accuracy_score(y_val, val_pred),
        "test_acc": accuracy_score(y_test, test_pred),
        "found_token_count": found_train + found_val + found_test,
        "missing_token_count": missing_train + missing_val + missing_test
    })

gensim_df = pd.DataFrame(gensim_results).sort_values("val_acc", ascending=False)
gensim_df

### Part C interpretation

Word2Vec gives each known word a dense vector. The review vector is created by averaging word vectors. This is fast and simple, but it loses word order.

Skipping OOV words is simple, but dangerous if an unknown word carries sentiment. Using an average vector keeps sequence information less empty, but it can also add neutral noise.


## Part D. Character-Based Classification

This model reads text as characters, not words.

Pipeline:

**text → characters → character ids → embedding → CNN → classifier**

A character model can handle misspellings and rare words better because it does not require the full word to exist in the vocabulary.


In [ ]:
# Character vocabulary from TRAIN ONLY

CHAR_PAD = "<PAD>"
CHAR_UNK = "<UNK>"
CHAR_PAD_IDX = 0
CHAR_UNK_IDX = 1

max_len_char = 700

char_counter = Counter(ch for text in train_texts for ch in text.lower())

char_stoi = {
    CHAR_PAD: CHAR_PAD_IDX,
    CHAR_UNK: CHAR_UNK_IDX
}

for ch, count in sorted(char_counter.items()):
    if count >= 2:
        char_stoi[ch] = len(char_stoi)

def encode_chars(text, char_stoi, max_len):
    ids = [char_stoi.get(ch, CHAR_UNK_IDX) for ch in text.lower()]

    if len(ids) < max_len:
        ids = ids + [CHAR_PAD_IDX] * (max_len - len(ids))
    else:
        ids = ids[:max_len]

    return ids

X_train_char = [encode_chars(t, char_stoi, max_len_char) for t in train_texts]
X_val_char = [encode_chars(t, char_stoi, max_len_char) for t in val_texts]
X_test_char = [encode_chars(t, char_stoi, max_len_char) for t in test_texts]

print("Character model setup:")
print("maximum character length:", max_len_char)
print("character vocabulary size:", len(char_stoi))

In [ ]:
class CharDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


class CharCNNClassifier(nn.Module):
    def __init__(self, vocab_size, emb_dim=64, num_filters=128, num_classes=2, pad_idx=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_idx)
        self.conv = nn.Conv1d(emb_dim, num_filters, kernel_size=5, padding=2)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(num_filters, num_classes)

    def forward(self, x):
        embedded = self.embedding(x)          # [batch, seq, emb]
        embedded = embedded.transpose(1, 2)   # [batch, emb, seq]
        conv_out = self.relu(self.conv(embedded))
        pooled = torch.max(conv_out, dim=2).values
        pooled = self.dropout(pooled)
        return self.fc(pooled)


batch_size_char = 64

train_loader_char = DataLoader(CharDataset(X_train_char, y_train), batch_size=batch_size_char, shuffle=True)
val_loader_char = DataLoader(CharDataset(X_val_char, y_val), batch_size=batch_size_char, shuffle=False)
test_loader_char = DataLoader(CharDataset(X_test_char, y_test), batch_size=batch_size_char, shuffle=False)

char_model = CharCNNClassifier(
    vocab_size=len(char_stoi),
    emb_dim=64,
    num_filters=128,
    num_classes=2,
    pad_idx=CHAR_PAD_IDX
).to(DEVICE)

criterion_char = nn.CrossEntropyLoss()
optimizer_char = torch.optim.Adam(char_model.parameters(), lr=0.001)

print(char_model)

In [ ]:
def train_one_epoch_char(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for x, y in loader:
        x = x.to(DEVICE)
        y = y.to(DEVICE)

        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * y.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == y).sum().item()
        total += y.size(0)

    return total_loss / total, correct / total


@torch.no_grad()
def evaluate_char(model, loader, criterion=None):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    all_preds = []
    all_targets = []

    for x, y in loader:
        x = x.to(DEVICE)
        y = y.to(DEVICE)

        logits = model(x)

        if criterion is not None:
            loss = criterion(logits, y)
            total_loss += loss.item() * y.size(0)

        preds = logits.argmax(dim=1)

        correct += (preds == y).sum().item()
        total += y.size(0)

        all_preds.extend(preds.cpu().numpy().tolist())
        all_targets.extend(y.cpu().numpy().tolist())

    avg_loss = total_loss / total if criterion is not None else None
    acc = correct / total

    return avg_loss, acc, np.array(all_preds), np.array(all_targets)


epochs_char = 3
history_char = []

for epoch in range(1, epochs_char + 1):
    train_loss, train_acc = train_one_epoch_char(char_model, train_loader_char, optimizer_char, criterion_char)
    val_loss, val_acc, _, _ = evaluate_char(char_model, val_loader_char, criterion_char)

    history_char.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "val_loss": val_loss,
        "val_acc": val_acc
    })

    print(
        f"Epoch {epoch}/{epochs_char} | "
        f"train_loss={train_loss:.4f}, train_acc={train_acc:.4f}, "
        f"val_loss={val_loss:.4f}, val_acc={val_acc:.4f}"
    )

train_loss_char, train_acc_char, train_preds_char, train_targets_char = evaluate_char(char_model, train_loader_char, criterion_char)
val_loss_char, val_acc_char, val_preds_char, val_targets_char = evaluate_char(char_model, val_loader_char, criterion_char)
test_loss_char, test_acc_char, test_preds_char, test_targets_char = evaluate_char(char_model, test_loader_char, criterion_char)

print("\nCharacter model accuracy:")
print("Train accuracy:", round(train_acc_char, 4))
print("Validation accuracy:", round(val_acc_char, 4))
print("Test accuracy:", round(test_acc_char, 4))

In [ ]:
# Examples where character model behaves differently from word model

diff_idx = [i for i in range(len(test_preds_char)) if test_preds_char[i] != test_preds_word[i]][:3]

print("3 examples where char model != word model:")
for i in diff_idx:
    print(f"Index {i} | char_pred={test_preds_char[i]} | word_pred={test_preds_word[i]} | true={y_test[i]}")
    print(test_texts[i][:320].replace("\n", " "), "...")
    print("-" * 80)

### Part D interpretation

The character model is usually more robust to noisy text, spelling variations, and rare words. However, it has to process longer sequences because every review becomes a sequence of characters instead of a sequence of words.

That is why it can be useful for noisy input, but it may not always beat a good word-level model on clean IMDB reviews.


## Part E. Compare All Approaches

Now I summarize all approaches in one comparison table.


In [ ]:
# Select the best Gensim strategy by validation accuracy
best_gensim = gensim_df.iloc[0]

comparison_rows = [
    {
        "Model / Representation": "Word IDs + learned embedding (GRU)",
        "Preprocessing": "simple lowercase + punctuation removal",
        "OOV Strategy": "<UNK> index",
        "Vocab Size": len(stoi),
        "Max Length": max_len_word,
        "Train Acc": round(float(train_acc_word), 4),
        "Val Acc": round(float(val_acc_word), 4),
        "Test Acc": round(float(test_acc_word), 4),
        "Notes": "Learns task-specific embeddings and uses word order"
    },
    {
        "Model / Representation": "Gensim Word2Vec + Logistic Regression",
        "Preprocessing": "simple tokens",
        "OOV Strategy": best_gensim["strategy"],
        "Vocab Size": len(w2v.wv),
        "Max Length": "mean pooled",
        "Train Acc": round(float(best_gensim["train_acc"]), 4),
        "Val Acc": round(float(best_gensim["val_acc"]), 4),
        "Test Acc": round(float(best_gensim["test_acc"]), 4),
        "Notes": "Fast and compact, but loses word order"
    },
    {
        "Model / Representation": "Character CNN",
        "Preprocessing": "lowercase raw chars",
        "OOV Strategy": "<UNK> char",
        "Vocab Size": len(char_stoi),
        "Max Length": max_len_char,
        "Train Acc": round(float(train_acc_char), 4),
        "Val Acc": round(float(val_acc_char), 4),
        "Test Acc": round(float(test_acc_char), 4),
        "Notes": "More robust to misspellings and rare words"
    }
]

comparison_df = pd.DataFrame(comparison_rows)
comparison_df

### Final comparison answers

**Which representation worked best?**  
The best representation is the one with the highest validation and test accuracy in the comparison table. In many IMDB experiments, Word2Vec with logistic regression can be very competitive, while the GRU word model is stronger conceptually because it uses sequence order.

**Which preprocessing choice mattered most?**  
The most important choice is preserving sentiment words and negation. Removing too many stopwords can hurt because words like **not** and **never** may change the meaning.

**Which unknown-word strategy worked best?**  
In Part C, the best OOV strategy is selected by validation accuracy. Usually, average-vector replacement and skipping OOV words are close if there are not too many missing words.

**Which model would I choose for many misspellings or rare words?**  
I would choose the character-based model because it reads text at the letter level and does not depend on full known words.


## Follow-Up Questions for Understanding

### 1. What is the difference between a token, a vocabulary item, and an embedding vector?

A **token** is a piece of text after tokenization, usually a word or character.  
A **vocabulary item** is a token that has been added to the vocabulary and assigned an integer ID.  
An **embedding vector** is a dense numerical representation of a vocabulary item.

Example:

```text
token: "good"
vocabulary item: "good" → 125
embedding vector: [0.14, -0.21, 0.08, ...]
```

### 2. Why should the vocabulary be built only from the training data?

Because using validation or test data while building the vocabulary creates data leakage. The model would indirectly see information from the data that is supposed to be unseen.

### 3. What does `<UNK>` represent?

`<UNK>` means unknown token. It is used for words that are not in the vocabulary, usually because they are rare or appear only in validation/test data.

### 4. Why can skipping unknown words be dangerous for sentiment classification?

Because an unknown word can contain important sentiment information. If the sentence is **not good** and the word **not** is skipped, the meaning changes completely.

### 5. Why might a character-based model handle unseen words better than a word-level model?

A character model reads letters, so it can still process unfamiliar words, misspellings, or unusual forms. A word-level model needs the whole word to exist in its vocabulary.

### 6. What information can be lost when representing a review by the mean of its word vectors?

Mean pooling loses word order and sentence structure. For example, **not good** and **good not** may become very similar after averaging.

### 7. Why can stopword removal sometimes hurt sentiment classification?

Because some stopwords are important for sentiment, especially negation words like **not**, **no**, and **never**.

### 8. Which approach was most sensitive to preprocessing?

The word-level model is very sensitive to preprocessing because tokenization directly changes the vocabulary, token IDs, and the sequences that the model receives.


## Conclusion

This notebook compared several ways to represent text for IMDB sentiment classification.

The word-level GRU model learns task-specific embeddings and uses word order.  
The Gensim Word2Vec model is simpler and fast, but mean pooling loses sequence information.  
The character model is useful when there are rare words, spelling mistakes, or noisy text.

Overall, the experiment shows that preprocessing, OOV handling, and representation choice strongly affect text classification performance.
